# Strands Agents com AgentCore Memory (Longo prazo) usando ferramentas

## Visão Geral
Este notebook demonstra como implementar capacidades de memória de longo prazo para agentes de IA conversacional usando Strands e AgentCore Memory. Você aprenderá como extrair e consolidar informações importantes de interações de curto prazo, permitindo que um agente recupere detalhes-chave em múltiplas sessões de conversa ao longo do tempo.

## Detalhes do Tutorial
**Caso de Uso:** Assistente Culinário com Memória Persistente

| Informação          | Detalhes                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| Tipo de tutorial    | Conversacional de longo prazo                                                    |
| Tipo de agente      | Assistente Culinário                                                             |
| Framework agêntico  | Strands Agents                                                                   |
| Modelo LLM          | Anthropic Claude Haiku 4.5                                                      |
| Componentes do tutorial | Extração de Memória 'Preferências do Usuário' do AgentCore, Ferramenta de Memória para armazenar e recuperar Memória |
| Complexidade do exemplo | Iniciante                                                                    |

Você aprenderá a:
- Configurar o AgentCore Memory com estratégias de extração para retenção de longo prazo
- Hidratar a memória com histórico de conversas anteriores
- Usar memória de longo prazo para oferecer experiências personalizadas entre sessões de conversa
- Integrar o Strands Agent Framework com a ferramenta AgentCore Memory

## Contexto do Cenário

Neste tutorial, você assumirá o papel de um Assistente Culinário projetado para fornecer recomendações de restaurantes altamente personalizadas. Ao aproveitar a retenção de longo prazo e a extração automática de informações do AgentCore Memory, o agente pode lembrar preferências do usuário — como escolhas alimentares e culinárias favoritas — em múltiplas conversas. Essa memória persistente permite que o agente forneça sugestões personalizadas e uma experiência de usuário fluida, mesmo quando as conversas se estendem por dias ou semanas. O cenário demonstra como a organização estruturada de memória e estratégias configuráveis capacitam a IA conversacional a ir além da memória de curto prazo, criando interações verdadeiramente envolventes e conscientes do contexto.


## Arquitetura

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## Pré-requisitos

Para executar este tutorial você precisará de:
- Python 3.10+
- Credenciais AWS com permissões do Amazon Bedrock AgentCore Memory
- SDK do Amazon Bedrock AgentCore
Vamos começar configurando nosso ambiente e criando nosso recurso de memória de longo prazo com a estratégia de extração apropriada!

## Passo 1: Configuração do ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para fazer este notebook funcionar.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import time
import logging
import time
from datetime import datetime

Defina a região e o perfil com as permissões apropriadas para modelos do Amazon Bedrock e AgentCore

In [ ]:
import os

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("culinary-memory")

region = os.getenv('AWS_REGION', 'us-west-2')

## Passo 2: Criando Memória com Estratégias de Longo Prazo

Nesta seção, criaremos um recurso de memória configurado com capacidades de memória de longo prazo. Diferentemente do nosso exemplo anterior de memória de curto prazo, esta implementação inclui estratégias de memória específicas que permitem a retenção consolidada de informações.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

client = MemoryClient(region_name=region)

memory_name = "CulinaryAssistant"
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Long-Term Memory...")

    # We use a more descriptive name for our long-term memory resource
    memory_name = memory_name

    # Create memory with user preference strategy
    memory = client.create_memory_and_wait(
        name=memory_name,
        description="Culinary Assistant Agent with long term memory",
        strategies=[{
                    StrategyType.USER_PREFERENCE.value: {
                        "name": "UserPreferences",
                        "description": "Captures user preferences",
                        "namespaceTemplates": ["user/{actorId}/preferences/"]
                    }
                }],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )

    memory_id = memory['id']
    print(f"Memory created successfully with ID: {memory_id}")
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    logger.info(f"❌ ERROR: {e}")
    import traceback
    traceback.print_exc()
    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Entendendo as Estratégias de Memória de Longo Prazo

A principal diferença nesta criação de memória é a adição de uma **estratégia de memória**. Vamos detalhar os componentes:

#### 1. Estratégia de Memória de Preferências do Usuário

Esta estratégia identifica e extrai automaticamente as preferências do usuário das conversas:

```python
"userPreferenceMemoryStrategy": {
    "name": "UserPreferences",
    "description": "Captures user preferences",
    "namespaceTemplates": ["user/{actorId}/preferences/"]
}
```

#### 2. Namespaces de Memória

O parâmetro `namespaceTemplates` define onde as informações extraídas são armazenadas:

```python
"namespaceTemplates": ["user/{actorId}/preferences/"]
```

Esta estratégia de memória cria um sistema de memória mais sofisticado que não apenas lembra conversas, mas realmente compreende e organiza as informações importantes dentro dessas conversas para uso futuro.

## Passo 3: Salvando Conversas Anteriores na Memória

Nesta seção, demonstraremos como hidratar a memória de curto prazo, o que automaticamente aciona o processo de extração de memória de longo prazo nos bastidores.

### Hidratando a Memória de Curto Prazo

Quando salvamos conversas em um recurso de memória configurado com estratégias de extração, o sistema processa automaticamente essas informações para retenção de longo prazo sem necessidade de código adicional.

In [ ]:
actor_id = f"user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"foodie-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"user/{actor_id}/preferences/"

In [ ]:
previous_messages = [
    ("Hi, I'm John", "USER"),
    ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
    ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
    ("That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?", "ASSISTANT"),
    ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
    ("Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?", "ASSISTANT"),
    ("I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.", "USER"),
    ("Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!", "ASSISTANT"),
    ("I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes", "USER"),
    ("I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?", "ASSISTANT"),
    ("Hey, I appreciate food assistants with good taste", "USER"),
    ("Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?", "ASSISTANT")
]

In [ ]:
print("\nHydrating short term memory with previous conversations...")

# Save the conversation history to short-term memory
initial = client.create_event(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    messages=previous_messages,
)
print("✓ Conversation saved in short term memory")

Vamos verificar se o evento contendo as mensagens da conversa foi armazenado corretamente.

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=5
)
events

Esta célula configura o sistema de logging para exibir mensagens informativas durante a execução, ajudando-nos a acompanhar o que está acontecendo enquanto nosso código é executado.

### O Que Acontece nos Bastidores

Após a chamada `create_event`, o seguinte ocorre automaticamente:

1. **Armazenamento de Curto Prazo**: A conversa completa é salva em formato bruto
2. **Gatilho de Extração**: O sistema de memória detecta que esta memória possui a estratégia UserPreference configurada
3. **Processamento em Segundo Plano**: Sem nenhum código adicional, o sistema:
   - Analisa a conversa em busca de indicadores de preferência
   - Identifica declarações como "Sou vegetariano" e "Eu realmente gosto de culinária italiana"
   - Extrai essas preferências em dados estruturados
4. **Consolidação de Longo Prazo**: As preferências extraídas são salvas no namespace configurado (`user/{actorId}/preferences/`)

A extração e consolidação acontecem automaticamente — precisamos apenas manter uma conversa com o agente ou hidratar a memória de curto prazo, e as estratégias que configuramos durante a criação da memória cuidam do resto.

Este processo automático garante que informações importantes sejam preservadas na memória de longo prazo mesmo após os registros de conversa de curto prazo expirarem.


## Recuperando Memórias de Longo Prazo

Nesta seção, exploraremos como acessar as preferências extraídas que foram armazenadas na memória de longo prazo. Diferentemente da recuperação de memória de curto prazo, que foca em turnos de conversa, a recuperação de memória de longo prazo foca em acessar informações estruturadas que foram extraídas e consolidadas.

### Acessando Preferências do Usuário na Memória de Longo Prazo

Para recuperar informações da memória de longo prazo, usamos a estrutura de namespace definida durante a criação da memória:


In [ ]:
# Adding a 30s wait to ensure the memory extraction has time to process the event
time.sleep(30)

try:
    # Query the memory system for food preferences
    food_preferences = client.retrieve_memories(
        memory_id=memory_id,
        namespace=namespace,
        query="food preferences",
        top_k=3  # Return up to 3 most relevant results
    )

    if food_preferences:
        print(f"Retrieved {len(food_preferences)} relevant preference records:")
        for i, record in enumerate(food_preferences):
            print(f"\nMemory {i+1}:")
            print(f"- Content: {record.get('content', 'Not specified')}")
    else:
        print("No matching preference records found.")

except Exception as e:
    print(f"Error retrieving preference records: {e}")

Este método permite a recuperação de memórias relevantes quando necessário. Agora que aprendemos o básico, vamos construir nosso agente!

## Passo 4: Criando o agente 
Nesta seção, exploraremos como integrar o AgentCore Memory com um Strands Agent usando a ferramenta nativa `agent_core_memory`.

#### Configurando o Agente com Capacidades de Memória de Longo Prazo
Para criar um agente com memória habilitada, usaremos o framework Strands e o conectaremos ao nosso recurso AgentCore Memory

In [ ]:
from strands import tool, Agent
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

In [ ]:
system_prompt = f"""You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.

PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide personalized dining recommendations

You have access to a Memory tool that enables you to:
- Store user preferences (dietary restrictions, favorite cuisines, budget preferences, etc.)
- Retrieve previously stored information to personalize recommendations

"""

In [ ]:
provider = AgentCoreMemoryToolProvider(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    namespace=namespace
)

agent = Agent(tools=provider.tools, model="global.anthropic.claude-haiku-4-5-20251001-v1:0",system_prompt=system_prompt)

Como já populamos nossa memória de curto e longo prazo, vamos recuperar diretamente a memória a partir do agente!

In [ ]:
agent("Give me restaurant recommendations in Irvine based on my food preferences")

O agente deveria ter usado o método retrieve_memory_records para recuperar as memórias do usuário.

Ótimo! Agora você tem um Strands Agent funcional capaz de recuperar memórias da Memória de Longo Prazo do AgentCore!

## Limpeza
Vamos excluir a memória para limpar os recursos utilizados neste notebook.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)